# RAG Indexing: Vector Databases (Qdrant)

## Overview
Vector databases are the storage and retrieval engine for dense embeddings in production RAG systems.
This notebook builds the **one corpus the whole RAG course uses**: **HotpotQA** multi-hop distractor passages, embedded and indexed into a Qdrant collection (`hotpotqa_passages`).

1. **Corpus and QA pairs**: Download HotpotQA, keep a fixed sample of questions and every candidate paragraph, and cache both.
2. **Indexing**: Embed the paragraphs and store them in Qdrant with title and gold-label payloads.
3. **First retrieval measurements**: Recall of the supporting paragraphs at k = 3, 5, 10.

Every later notebook (6.6 to 6.13) reads this index and these caches through `rag_common.py`.

> [!NOTE]
> Qdrant must be running. From `code/week6`: `docker compose up -d` (or `docker run -p 6333:6333 qdrant/qdrant`).

In [ ]:
%pip install -q datasets langchain langchain-qdrant langchain-huggingface sentence-transformers qdrant-client pyyaml tqdm


In [ ]:
import json
import random
from statistics import mean
from tqdm import tqdm

from datasets import load_dataset
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

import rag_common as rc


## 1. Load Configuration & Initialize Embeddings

All connection endpoints, model names, and dataset paths are centralized in `config.yaml`.


In [ ]:
cfg = rc.load_config()

QDRANT_URL   = cfg["qdrant"]["url"]
EMBED_MODEL  = cfg["embeddings"]["model"]

# HotpotQA config
hcfg           = cfg["hotpotqa"]
HP_COLLECTION  = hcfg["collection"]
HP_SPLIT       = hcfg.get("split", "validation")
HP_SAMPLE      = hcfg.get("sample_size", 100)
HP_SEED        = hcfg.get("random_seed", 42)
HP_QA_CACHE    = hcfg["qa_pairs_cache"]
HP_PASS_CACHE  = hcfg["passages_cache"]

rc.check_services(cfg, llm=False)
print(f"Qdrant URL      : {QDRANT_URL}")
print(f"Embedding Model : {EMBED_MODEL}")

# Initialize shared embedding model & Qdrant client
embeddings  = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
vector_size = len(embeddings.embed_query("test"))
client      = QdrantClient(url=QDRANT_URL)
print(f"Vector dimension: {vector_size}")


---
## 2. The Corpus: HotpotQA (`hotpotqa_passages`)

We use the **distractor** configuration of HotpotQA — each question comes bundled with 10 paragraphs:
2 gold supporting paragraphs + 8 distractors from other Wikipedia articles.

This provides a challenging benchmark where the model must synthesize facts across multiple passages to answer the query.


In [ ]:
print(f"Loading HotpotQA (distractor / {HP_SPLIT}) …")
ds = load_dataset("hotpot_qa", "distractor", split=HP_SPLIT)
print(f"Downloaded {len(ds)} examples.")

random.seed(HP_SEED)
indices = random.sample(range(len(ds)), k=min(HP_SAMPLE, len(ds)))
sample  = [ds[i] for i in indices]

# --- QA pairs ---
hp_qa_pairs = [
    {
        "question":          ex["question"],
        "answer":            ex["answer"],
        "type":              ex["type"],
        "level":             ex["level"],
        "supporting_titles": list(dict.fromkeys(ex["supporting_facts"]["title"])),
    }
    for ex in sample
]

# --- Corpus of all candidate paragraphs ---
seen_keys = set()
hp_corpus    = []
for ex in sample:
    supp_set = set(ex["supporting_facts"]["title"])
    for title, sentences in zip(ex["context"]["title"], ex["context"]["sentences"]):
        key = (title, 0)
        if key not in seen_keys:
            seen_keys.add(key)
            text = " ".join(sentences).strip()
            if text:
                hp_corpus.append({
                    "id":            f"{title}#0",
                    "title":         title,
                    "text":          text,
                    "is_supporting": title in supp_set,
                })

with open(HP_QA_CACHE, "w") as f:
    json.dump(hp_qa_pairs, f)
print(f"Saved {len(hp_qa_pairs)} QA pairs → {HP_QA_CACHE}")

with open(HP_PASS_CACHE, "w") as f:
    json.dump(hp_corpus, f)
print(f"Saved {len(hp_corpus)} passages → {HP_PASS_CACHE}")


In [ ]:
if client.collection_exists(HP_COLLECTION):
    print(f"Dropping existing collection '{HP_COLLECTION}' …")
    client.delete_collection(HP_COLLECTION)

client.create_collection(
    collection_name=HP_COLLECTION,
    vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE),
)

hp_doc_store = QdrantVectorStore(
    client=client, collection_name=HP_COLLECTION, embedding=embeddings
)

BATCH = 64
texts     = [p["text"] for p in hp_corpus]
metadatas = [{"title": p["title"], "is_supporting": p["is_supporting"]} for p in hp_corpus]

for i in tqdm(range(0, len(texts), BATCH), desc="Indexing HotpotQA"):
    hp_doc_store.add_texts(texts=texts[i:i+BATCH], metadatas=metadatas[i:i+BATCH])

n = client.get_collection(HP_COLLECTION).points_count
print(f"Indexed {n} paragraphs into '{HP_COLLECTION}'.")


In [ ]:
rc.reset_ablation()   # a fresh index starts a fresh results ledger (see rc.show_ablation() in 6.11)

In [ ]:
K_VALUES = [3, 5, 10]

print(f"{'k':>4}  {'Both found':>12}  {'At least 1':>12}  {'Avg recall':>12}")
print("-" * 46)

for k in K_VALUES:
    ret = hp_doc_store.as_retriever(search_kwargs={"k": k})
    both_found = []
    at_least_1 = []
    recalls    = []

    for qa in hp_qa_pairs:
        docs     = ret.invoke(qa["question"])
        ret_titles = set(d.metadata["title"] for d in docs)
        supp     = set(qa["supporting_titles"])
        overlap  = len(ret_titles & supp)
        recalls.append(overlap / len(supp) if supp else 1.0)
        both_found.append(int(supp.issubset(ret_titles)))
        at_least_1.append(int(overlap > 0))

    print(
        f"{k:>4}  "
        f"{mean(both_found)*100:>11.1f}%  "
        f"{mean(at_least_1)*100:>11.1f}%  "
        f"{mean(recalls)*100:>11.1f}%"
    )


---
## Summary & Key Takeaways — Vector Databases

1. **Dense Vector Indexing**: Qdrant stores high-dimensional embeddings and delivers sub-millisecond approximate nearest neighbor (ANN) retrieval using HNSW graphs.
2. **Payloads & Metadata**: Attaching titles and gold labels directly to vectors enables filtered retrieval and evaluation without a second database.
3. **One corpus for the course**: `hotpotqa_passages` holds multi-hop questions' paragraphs (2 gold + 8 distractors each). Every later RAG notebook measures on the same fixed question sample, so results are comparable.
4. **Next Step**: Proceed to **`6.6-Storage-Chunking-Strategies.ipynb`** to see how chunking changes retrieval.